# Part 3: Sales Forecasting Model — DATATHON 2026
**Team:** GenCore
**Lead:** Trịnh Hoàng Tú

**Objective:** Predict daily `Revenue` and `COGS` for the period 2023-01-01 to 2024-07-01.

**Strategy:** 
1. **Baseline**: Seasonal profile scaled by YoY growth.
2. **Advanced**: XGBoost Regressor incorporating web traffic and time-based features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.preprocessing import load_and_merge_order_data

DATA_PATH = '../data/raw/'
RANDOM_SEED = 42
VND_RATE = 25450

## 1 — Data Loading

In [ ]:
sales = pd.read_csv(os.path.join(DATA_PATH, 'sales.csv'), parse_dates=['Date'])
web_traffic = pd.read_csv(os.path.join(DATA_PATH, 'web_traffic.csv'), parse_dates=['date'])
sample_sub = pd.read_csv(os.path.join(DATA_PATH, 'sample_submission.csv'), parse_dates=['Date'])

print(f"Sales range: {sales['Date'].min()} to {sales['Date'].max()}")
print(f"Test range: {sample_sub['Date'].min()} to {sample_sub['Date'].max()}")

## 2 — Feature Engineering
We build features while strictly avoiding data leakage.

In [ ]:
def create_advanced_features(df, traffic=None):
    df = df.copy()
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['dayofyear'] = df['Date'].dt.dayofyear
    df['weekofyear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    # Holiday Indicators (Approximation for VN context)
    # Tet (approx Jan/Feb), Black Friday (Nov), etc.
    df['is_tet_season'] = df['month'].isin([1, 2]).astype(int)
    df['is_sale_month'] = df['month'].isin([11, 12]).astype(int)
    
    if traffic is not None:
        df = df.merge(traffic[['date', 'sessions', 'conversion_rate', 'bounce_rate']], 
                      left_on='Date', right_on='date', how='left')
        df.drop(columns=['date'], inplace=True)
        # Fill NaN for test period or missing traffic data
        df['sessions'] = df['sessions'].ffill().bfill()
        df['conversion_rate'] = df['conversion_rate'].ffill().bfill()
        df['bounce_rate'] = df['bounce_rate'].ffill().bfill()
        
    return df

full_df = pd.concat([sales, sample_sub[['Date']]], axis=0).sort_values('Date')
X = create_advanced_features(full_df, web_traffic)
X.head()

## 3 — Modeling: XGBoost Regressor
We train on individual targets: `Revenue` and `COGS`.

In [ ]:
train_idx = X['Date'] <= sales['Date'].max()
test_idx = X['Date'] > sales['Date'].max()

features = ['month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 
            'is_weekend', 'is_tet_season', 'is_sale_month', 
            'sessions', 'conversion_rate', 'bounce_rate']

X_train = X.loc[train_idx, features]
X_test = X.loc[test_idx, features]

y_rev = sales['Revenue']
y_cogs = sales['COGS']

# Model for Revenue
model_rev = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=RANDOM_SEED)
model_rev.fit(X_train, y_rev)

# Model for COGS
model_cogs = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=RANDOM_SEED)
model_cogs.fit(X_train, y_cogs)

print("Models trained successfully.")

## 4 — Predictions & Submission

In [ ]:
rev_pred = model_rev.predict(X_test)
cogs_pred = model_cogs.predict(X_test)

submission = sample_sub.copy()
submission['Revenue'] = np.maximum(0, rev_pred).round(2)
submission['COGS'] = np.maximum(0, cogs_pred).round(2)

if not os.path.exists('../output'):
    os.makedirs('../output')
    
submission.to_csv('../output/submission_tu.csv', index=False)
print("Leader's submission saved to output/submission_tu.csv")
submission.head()